# Parameter search (Optuna)

Searches genetic algorithm parameters with Optuna on training layouts from `data/training/`.
Two searches are run:

1. **ALL** - one study over all selected training cases.
2. **By category** - independent studies for the `short`, `medium` and `tall` size categories.

Results are saved to `parameter_search/artifacts/` and later used by `run_genetic_algorithm_auto`.
Run the cells from top to bottom. The searches can take a long time.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# Find the project root (the directory that contains the parameter_search/ package).
ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "parameter_search").is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from parameter_search.search import (
    load_and_select_training_cases,
    search_parameters_all,
    search_parameters_by_category,
)

ARTIFACTS = ROOT / "parameter_search/artifacts"
PER_CATEGORY = 50  # training cases picked per size category

SEARCH = dict(
    n_trials=50,
    packer="skyline",
    time_weight=0.10,
    time_reference_seconds=3.0,
    max_population_size=130,
    max_generations=180,
)

c:\Users\PC\my-projects\corel-layout-optimizer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def show_results(rows, title):
    """Show top trials by height quality and by quality + time; return the best of each."""
    df = pd.DataFrame(rows)
    if df.empty:
        return None, None
    by_quality = df.sort_values("quality_objective").reset_index(drop=True)
    by_value = df.sort_values("objective").reset_index(drop=True)
    print(f"\n{title} - quality without time:")
    display(by_quality.head(10))
    print(f"{title} - full score: quality + time:")
    display(by_value.head(10))
    best_quality, best_value = by_quality.iloc[0].to_dict(), by_value.iloc[0].to_dict()
    print("Best quality:", best_quality)
    print("Best value:", best_value)
    return best_quality, best_value

## Training cases

In [3]:
training_cases = load_and_select_training_cases(ROOT, per_category=PER_CATEGORY)
print(f"Training cases selected: {len(training_cases)}")
print(pd.Series([case["size_category"] for case in training_cases]).value_counts().sort_index())

Training cases selected: 150
medium    50
short     50
tall      50
Name: count, dtype: int64


## Search over all cases

In [4]:
rows, _ = search_parameters_all(
    training_cases,
    results_path=ARTIFACTS / "parameter_search_all_3s_results.json",
    study_path=ARTIFACTS / "parameter_search_all_3s.db",
    **SEARCH,
)
best_quality_parameters, best_value_parameters = show_results(rows, "ALL")

[I 2026-11-12 17:46:42,868] Using an existing study with `study_name='layout_optimizer'` instead of creating a new one.



ALL - quality without time:


,trial,population_size,generations,mutation_rate,stagnation_generations,elite_count,crossover_probability,orientation_mutation_rate,swap_mutation_rate,min_width_ratio,...,mean_seconds,time_penalty,objective,state,short_quality_objective,short_mean_seconds,medium_quality_objective,medium_mean_seconds,tall_quality_objective,tall_mean_seconds
0,3,100,30,0.283368,30,2,0.564463,0.092549,0.250041,0.816675,...,1.126412,0.018774,0.886970,COMPLETE,0.986323,0.319196,0.971937,0.958599,0.992051,2.101441
1,0,80,140,0.260967,25,3,0.822947,0.120645,0.540475,0.966192,...,1.772460,0.029541,0.890099,COMPLETE,0.989908,0.429843,0.971173,1.599568,0.996070,3.287970
2,4,90,60,0.213209,10,2,0.555188,0.170956,0.126001,0.674701,...,0.538583,0.008976,0.888967,COMPLETE,0.981714,0.130797,0.973171,0.525307,1.005347,0.959647
3,2,80,90,0.142594,35,3,0.784217,0.024322,0.389700,0.832596,...,1.352632,0.022544,0.897663,COMPLETE,1.005659,0.242575,0.982098,1.235632,0.996939,2.579688
4,1,20,30,0.057076,35,5,0.935006,0.245082,0.489537,0.775362,...,0.133471,0.002225,0.906546,COMPLETE,1.005914,0.039657,0.998167,0.131323,1.017000,0.229432


ALL - full score: quality + time:


,trial,population_size,generations,mutation_rate,stagnation_generations,elite_count,crossover_probability,orientation_mutation_rate,swap_mutation_rate,min_width_ratio,...,mean_seconds,time_penalty,objective,state,short_quality_objective,short_mean_seconds,medium_quality_objective,medium_mean_seconds,tall_quality_objective,tall_mean_seconds
0,3,100,30,0.283368,30,2,0.564463,0.092549,0.250041,0.816675,...,1.126412,0.018774,0.886970,COMPLETE,0.986323,0.319196,0.971937,0.958599,0.992051,2.101441
1,4,90,60,0.213209,10,2,0.555188,0.170956,0.126001,0.674701,...,0.538583,0.008976,0.888967,COMPLETE,0.981714,0.130797,0.973171,0.525307,1.005347,0.959647
2,0,80,140,0.260967,25,3,0.822947,0.120645,0.540475,0.966192,...,1.772460,0.029541,0.890099,COMPLETE,0.989908,0.429843,0.971173,1.599568,0.996070,3.287970
3,2,80,90,0.142594,35,3,0.784217,0.024322,0.389700,0.832596,...,1.352632,0.022544,0.897663,COMPLETE,1.005659,0.242575,0.982098,1.235632,0.996939,2.579688
4,1,20,30,0.057076,35,5,0.935006,0.245082,0.489537,0.775362,...,0.133471,0.002225,0.906546,COMPLETE,1.005914,0.039657,0.998167,0.131323,1.017000,0.229432


Best quality: {'trial': 3, 'population_size': 100, 'generations': 30, 'mutation_rate': 0.28336835040598374, 'stagnation_generations': 30, 'elite_count': 2, 'crossover_probability': 0.5644631488274267, 'orientation_mutation_rate': 0.0925485207125623, 'swap_mutation_rate': 0.2500409240184424, 'min_width_ratio': 0.8166747727587942, 'mix_categories': 1, 'epsilon': 0.19790729085066072, 'prefer_higher': True, 'acceptance': 0.8322619035769993, 'quality_objective': 0.9834367182838732, 'mean_seconds': 1.1264121080000344, 'time_penalty': 0.018773535133333907, 'objective': 0.8869703999688193, 'state': 'COMPLETE', 'short_quality_objective': 0.9863228663193155, 'short_mean_seconds': 0.3191963880000003, 'medium_quality_objective': 0.9719365754965483, 'medium_mean_seconds': 0.9585991020000256, 'tall_quality_objective': 0.9920507130357558, 'tall_mean_seconds': 2.101440834000077}
Best value: {'trial': 3, 'population_size': 100, 'generations': 30, 'mutation_rate': 0.28336835040598374, 'stagnation_genera

## Search per size category (short / medium / tall)

In [5]:
category_results = search_parameters_by_category(
    training_cases,
    results_path=ARTIFACTS / "parameter_search_category_3s_results.json",
    study_path=ARTIFACTS / "parameter_search_category_3s.db",
    **SEARCH,
)
for category, (rows, _) in category_results.items():
    show_results(rows, category.upper())

[I 2026-11-12 17:46:44,905] Using an existing study with `study_name='layout_optimizer'` instead of creating a new one.
[I 2026-11-12 17:46:44,988] Using an existing study with `study_name='layout_optimizer'` instead of creating a new one.
[I 2026-11-12 17:46:45,063] Using an existing study with `study_name='layout_optimizer'` instead of creating a new one.



SHORT - quality without time:


,trial,population_size,generations,mutation_rate,stagnation_generations,elite_count,crossover_probability,orientation_mutation_rate,swap_mutation_rate,min_width_ratio,...,epsilon,prefer_higher,acceptance,quality_objective,mean_seconds,time_penalty,objective,state,short_quality_objective,short_mean_seconds
0,4,90,60,0.213209,10,2,0.555188,0.170956,0.126001,0.674701,...,0.167779,True,0.819220,0.981714,0.123768,0.002063,0.883749,COMPLETE,0.981714,0.123768
1,3,100,30,0.283368,30,2,0.564463,0.092549,0.250041,0.816675,...,0.197907,True,0.832262,0.986323,0.280374,0.004673,0.888158,COMPLETE,0.986323,0.280374
2,0,80,140,0.260967,25,3,0.822947,0.120645,0.540475,0.966192,...,0.162511,True,0.985119,0.989908,0.386173,0.006436,0.891561,COMPLETE,0.989908,0.386173
3,2,80,90,0.142594,35,3,0.784217,0.024322,0.389700,0.832596,...,0.189875,False,0.887406,1.005659,0.241619,0.004027,0.905495,COMPLETE,1.005659,0.241619
4,1,20,30,0.057076,35,5,0.935006,0.245082,0.489537,0.775362,...,0.041289,False,0.988934,1.005914,0.035906,0.000598,0.905382,COMPLETE,1.005914,0.035906


SHORT - full score: quality + time:


,trial,population_size,generations,mutation_rate,stagnation_generations,elite_count,crossover_probability,orientation_mutation_rate,swap_mutation_rate,min_width_ratio,...,epsilon,prefer_higher,acceptance,quality_objective,mean_seconds,time_penalty,objective,state,short_quality_objective,short_mean_seconds
0,4,90,60,0.213209,10,2,0.555188,0.170956,0.126001,0.674701,...,0.167779,True,0.819220,0.981714,0.123768,0.002063,0.883749,COMPLETE,0.981714,0.123768
1,3,100,30,0.283368,30,2,0.564463,0.092549,0.250041,0.816675,...,0.197907,True,0.832262,0.986323,0.280374,0.004673,0.888158,COMPLETE,0.986323,0.280374
2,0,80,140,0.260967,25,3,0.822947,0.120645,0.540475,0.966192,...,0.162511,True,0.985119,0.989908,0.386173,0.006436,0.891561,COMPLETE,0.989908,0.386173
3,1,20,30,0.057076,35,5,0.935006,0.245082,0.489537,0.775362,...,0.041289,False,0.988934,1.005914,0.035906,0.000598,0.905382,COMPLETE,1.005914,0.035906
4,2,80,90,0.142594,35,3,0.784217,0.024322,0.389700,0.832596,...,0.189875,False,0.887406,1.005659,0.241619,0.004027,0.905495,COMPLETE,1.005659,0.241619


Best quality: {'trial': 4, 'population_size': 90, 'generations': 60, 'mutation_rate': 0.2132087704997072, 'stagnation_generations': 10, 'elite_count': 2, 'crossover_probability': 0.5551875705821525, 'orientation_mutation_rate': 0.17095580557701287, 'swap_mutation_rate': 0.12600062324173758, 'min_width_ratio': 0.6747012974384203, 'mix_categories': 1, 'epsilon': 0.16777878137262833, 'prefer_higher': True, 'acceptance': 0.8192196815787927, 'quality_objective': 0.9817144683856731, 'mean_seconds': 0.12376781600005415, 'time_penalty': 0.0020627969333342363, 'objective': 0.8837493012404393, 'state': 'COMPLETE', 'short_quality_objective': 0.9817144683856731, 'short_mean_seconds': 0.12376781600005415}
Best value: {'trial': 4, 'population_size': 90, 'generations': 60, 'mutation_rate': 0.2132087704997072, 'stagnation_generations': 10, 'elite_count': 2, 'crossover_probability': 0.5551875705821525, 'orientation_mutation_rate': 0.17095580557701287, 'swap_mutation_rate': 0.12600062324173758, 'min_wid

... (4 outputs left out)

,trial,population_size,generations,mutation_rate,stagnation_generations,elite_count,crossover_probability,orientation_mutation_rate,swap_mutation_rate,min_width_ratio,...,epsilon,prefer_higher,acceptance,quality_objective,mean_seconds,time_penalty,objective,state,tall_quality_objective,tall_mean_seconds
0,3,100,30,0.283368,30,2,0.564463,0.092549,0.250041,0.816675,...,0.197907,True,0.832262,0.992051,2.005871,0.033431,0.896189,COMPLETE,0.992051,2.005871
1,0,80,140,0.260967,25,3,0.822947,0.120645,0.540475,0.966192,...,0.162511,True,0.985119,0.996070,3.046326,0.050238,0.901487,COMPLETE,0.996070,3.046326
2,2,80,90,0.142594,35,3,0.784217,0.024322,0.389700,0.832596,...,0.189875,False,0.887406,0.996939,2.632583,0.043876,0.901633,COMPLETE,0.996939,2.632583
3,5,130,90,0.391866,25,4,0.519594,0.085046,0.116108,0.712533,...,0.077237,False,0.938494,1.000468,2.619128,0.043652,0.904787,COMPLETE,1.000468,2.619128
4,39,100,90,0.285661,30,3,0.571295,0.054654,0.199162,0.676229,...,0.052547,False,0.895872,1.002685,1.539944,0.025666,0.904983,COMPLETE,1.002685,1.539944
5,35,90,90,0.137796,35,2,0.722473,0.038872,0.226776,0.763084,...,0.179439,False,0.950643,1.003432,2.679673,0.044661,0.907555,COMPLETE,1.003432,2.679673
6,4,90,60,0.213209,10,2,0.555188,0.170956,0.126001,0.674701,...,0.167779,True,0.819220,1.005347,0.954321,0.015905,0.906403,COMPLETE,1.005347,0.954321
7,47,80,90,0.084008,35,3,0.715985,0.025071,0.361443,0.847665,...,0.154014,False,0.849202,1.005651,1.929126,0.032152,0.908301,COMPLETE,1.005651,1.929126
8,9,20,70,0.281061,15,4,0.714384,0.051159,0.214055,0.816587,...,0.123379,False,0.886284,1.013816,0.310978,0.005183,0.912953,COMPLETE,1.013816,0.310978
9,1,20,30,0.057076,35,5,0.935006,0.245082,0.489537,0.775362,...,0.041289,False,0.988934,1.017000,0.231352,0.003856,0.915685,COMPLETE,1.017000,0.231352


TALL - full score: quality + time:


,trial,population_size,generations,mutation_rate,stagnation_generations,elite_count,crossover_probability,orientation_mutation_rate,swap_mutation_rate,min_width_ratio,...,epsilon,prefer_higher,acceptance,quality_objective,mean_seconds,time_penalty,objective,state,tall_quality_objective,tall_mean_seconds
0,3,100,30,0.283368,30,2,0.564463,0.092549,0.250041,0.816675,...,0.197907,True,0.832262,0.992051,2.005871,0.033431,0.896189,COMPLETE,0.992051,2.005871
1,0,80,140,0.260967,25,3,0.822947,0.120645,0.540475,0.966192,...,0.162511,True,0.985119,0.996070,3.046326,0.050238,0.901487,COMPLETE,0.996070,3.046326
2,2,80,90,0.142594,35,3,0.784217,0.024322,0.389700,0.832596,...,0.189875,False,0.887406,0.996939,2.632583,0.043876,0.901633,COMPLETE,0.996939,2.632583
3,5,130,90,0.391866,25,4,0.519594,0.085046,0.116108,0.712533,...,0.077237,False,0.938494,1.000468,2.619128,0.043652,0.904787,COMPLETE,1.000468,2.619128
4,39,100,90,0.285661,30,3,0.571295,0.054654,0.199162,0.676229,...,0.052547,False,0.895872,1.002685,1.539944,0.025666,0.904983,COMPLETE,1.002685,1.539944
5,4,90,60,0.213209,10,2,0.555188,0.170956,0.126001,0.674701,...,0.167779,True,0.819220,1.005347,0.954321,0.015905,0.906403,COMPLETE,1.005347,0.954321
6,35,90,90,0.137796,35,2,0.722473,0.038872,0.226776,0.763084,...,0.179439,False,0.950643,1.003432,2.679673,0.044661,0.907555,COMPLETE,1.003432,2.679673
7,47,80,90,0.084008,35,3,0.715985,0.025071,0.361443,0.847665,...,0.154014,False,0.849202,1.005651,1.929126,0.032152,0.908301,COMPLETE,1.005651,1.929126
8,9,20,70,0.281061,15,4,0.714384,0.051159,0.214055,0.816587,...,0.123379,False,0.886284,1.013816,0.310978,0.005183,0.912953,COMPLETE,1.013816,0.310978
9,1,20,30,0.057076,35,5,0.935006,0.245082,0.489537,0.775362,...,0.041289,False,0.988934,1.017000,0.231352,0.003856,0.915685,COMPLETE,1.017000,0.231352


Best quality: {'trial': 3, 'population_size': 100, 'generations': 30, 'mutation_rate': 0.28336835040598374, 'stagnation_generations': 30, 'elite_count': 2, 'crossover_probability': 0.5644631488274267, 'orientation_mutation_rate': 0.0925485207125623, 'swap_mutation_rate': 0.2500409240184424, 'min_width_ratio': 0.8166747727587942, 'mix_categories': 1, 'epsilon': 0.19790729085066072, 'prefer_higher': True, 'acceptance': 0.8322619035769993, 'quality_objective': 0.9920507130357558, 'mean_seconds': 2.0058708280000612, 'time_penalty': 0.03343118046666769, 'objective': 0.8961887597788469, 'state': 'COMPLETE', 'tall_quality_objective': 0.9920507130357558, 'tall_mean_seconds': 2.0058708280000612}
Best value: {'trial': 3, 'population_size': 100, 'generations': 30, 'mutation_rate': 0.28336835040598374, 'stagnation_generations': 30, 'elite_count': 2, 'crossover_probability': 0.5644631488274267, 'orientation_mutation_rate': 0.0925485207125623, 'swap_mutation_rate': 0.2500409240184424, 'min_width_rat